<a href="https://colab.research.google.com/github/bonnimv/bank-marketing-data/blob/main/notebooks/bank_marketing_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Imports ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print("✅ Bibliothèques chargées")

In [ ]:
# ── Rechargement du dataset ────────────────────────────────────────
# Règle d'or : on ne touche jamais aux données brutes !
# On repart toujours de df_raw et on travaille sur une copie.

from google.colab import files
uploaded = files.upload()  # re-uploader bank.csv

df_raw = pd.read_csv('bank.csv')
df_projet = df_raw.copy()

print(f"✅ {df_projet.shape[0]:,} clients — {df_projet.shape[1]} colonnes")
df_projet.head(3)

### 🔧 Séance 2 — Data Engineering & Préparation des données

Le pipeline de données suit toujours 3 étapes dans cet ordre :

1. **Collecte** — récupérer les données brutes (on l'a fait en séance 1)
2. **Nettoyage** — traiter les problèmes : valeurs manquantes,
   doublons, valeurs aberrantes
3. **Transformation** — convertir les textes en chiffres,
   mettre toutes les variables à la même échelle

> 💼 En langage métier : avant de donner les dossiers clients
> à un algorithme, il faut s'assurer qu'ils sont tous complets,
> cohérents, et rédigés dans un langage que la machine comprend.

In [ ]:
# ── 2.1 Traitement des valeurs manquantes et "unknown" ─────────────

# Rappel séance 1 : pas de NaN techniques, mais des "unknown"
# On choisit la stratégie pour chaque colonne

print("=== Nombre de 'unknown' par colonne ===")
for col in df_projet.select_dtypes(include='object').columns:
    nb_unk = (df_projet[col] == 'unknown').sum()
    pct_unk = round(nb_unk / len(df_projet) * 100, 1)
    if nb_unk > 0:
        print(f"{col:<15} : {nb_unk:>5} ({pct_unk}%)")

In [ ]:
# ── 2.2 Application de la stratégie ───────────────────────────────

# job (0.6%) → on remplace par le mode (valeur la plus fréquente)
# car c'est une variable catégorielle avec très peu de manquants
mode_job = df_projet[df_projet['job'] != 'unknown']['job'].mode()[0]
df_projet['job'] = df_projet['job'].replace('unknown', mode_job)
print(f"job     → 'unknown' remplacés par le mode : '{mode_job}'")

# education (4.5%) → on remplace par le mode
mode_edu = df_projet[df_projet['education'] != 'unknown']['education'].mode()[0]
df_projet['education'] = df_projet['education'].replace('unknown', mode_edu)
print(f"education → 'unknown' remplacés par le mode : '{mode_edu}'")

# contact (21%) → on garde 'unknown' comme catégorie à part entière
# car 21% c'est trop pour supprimer, et "moyen de contact inconnu"
# est une information en soi
print("contact → 'unknown' conservé comme catégorie")

# poutcome (74.6%) → on remplace par 'none'
# car ces clients n'ont jamais été contactés avant
df_projet['poutcome'] = df_projet['poutcome'].replace('unknown', 'none')
print("poutcome → 'unknown' remplacé par 'none' (jamais contacté avant)")

print()
print("=== Vérification après traitement ===")
for col in ['job', 'education', 'contact', 'poutcome']:
    nb_unk = (df_projet[col] == 'unknown').sum()
    print(f"{col:<15} : {nb_unk} 'unknown' restants")

### 🧹 2.1 & 2.2 — Traitement des valeurs manquantes

Notre dataset ne contient pas de NaN au sens technique, mais
des "unknown" qui masquent des informations absentes.

On applique une stratégie différente selon le cas :

- **`job` (0.6%)** et **`education` (4.5%)** : taux faible →
  on remplace par la valeur la plus fréquente (le mode).
  Garder "unknown" créerait une fausse catégorie.

- **`contact` (21%)** : taux trop élevé pour supprimer →
  on le conserve comme catégorie à part entière.
  "Moyen de contact inconnu" est une information en soi.

- **`poutcome` (74.6%)** : ces clients n'ont tout simplement
  jamais été contactés avant → on renomme "unknown" en "none"
  pour être plus explicite.

> 💼 En langage métier : on ne supprime pas un dossier client
> parce qu'il manque une info. On fait au mieux avec ce qu'on a,
> en étant transparent sur ce qu'on ne sait pas.

In [ ]:
# ── 3.1 Vérification des doublons ─────────────────────────────────

# Doublons exacts (toutes colonnes identiques)
nb_dup_exacts = df_projet.duplicated().sum()
print(f"Doublons exacts : {nb_dup_exacts}")

# Doublons sur une clé métier (même profil client)
nb_dup_metier = df_projet.duplicated(
    subset=['age', 'job', 'marital', 'education', 'balance']
).sum()
print(f"Doublons sur profil client : {nb_dup_metier}")
print()
print("→ Aucune suppression nécessaire.")

In [ ]:
# ── 3.2 Filtrage des outliers (règle 1.5×IQR) ─────────────────────

# Variable : campaign (nombre d'appels)
q1_c = df_projet['campaign'].quantile(0.25)
q3_c = df_projet['campaign'].quantile(0.75)
iqr_c = q3_c - q1_c
borne_sup_campaign = q3_c + 1.5 * iqr_c

# Variable : balance (solde bancaire)
q1_b = df_projet['balance'].quantile(0.25)
q3_b = df_projet['balance'].quantile(0.75)
iqr_b = q3_b - q1_b
borne_sup_balance = q3_b + 1.5 * iqr_b
borne_inf_balance = q1_b - 1.5 * iqr_b

print(f"Campaign  → borne sup : {borne_sup_campaign}")
print(f"Balance   → borne inf : {borne_inf_balance:.0f}€  |  borne sup : {borne_sup_balance:.0f}€")
print()

# Création de df_clean
df_clean = df_projet[
    (df_projet['campaign'] <= borne_sup_campaign) &
    (df_projet['balance'] >= borne_inf_balance) &
    (df_projet['balance'] <= borne_sup_balance)
].copy()

nb_clean = len(df_clean)
pct_clean = round(nb_clean / len(df_projet) * 100, 1)
print(f"Clients conservés dans df_clean : {nb_clean:,} ({pct_clean}%)")

### 🧹 3.1 & 3.2 — Doublons et outliers

**Doublons :** aucun doublon exact détecté. Chaque ligne
correspond bien à un client unique dans la campagne.

**Outliers :** on applique la règle 1.5×IQR sur deux variables :
- `campaign` : on retire les clients contactés trop de fois
  (plus de 6 appels — ces observations correspondent à des cas atypiques de sollicitation répétée)
- `balance` : on retire les soldes extrêmes dans les deux sens

On conserve **9 556 clients (85.6%)** — un taux de rétention
très satisfaisant qui garantit que nos conclusions restent
représentatives de la clientèle réelle.

> 💼 En langage métier : on écarte les cas trop atypiques
> pour que le modèle apprenne sur des clients "normaux",
> pas sur des exceptions qui fausseraient ses prédictions.

In [ ]:
# ── 4.1 Encodage de la variable cible ─────────────────────────────
# deposit : yes → 1, no → 0
# C'est une variable binaire (2 valeurs) → Label Encoding simple

df_clean['deposit_encoded'] = df_clean['deposit'].map({'yes': 1, 'no': 0})

print("=== Vérification deposit_encoded ===")
print(df_clean[['deposit', 'deposit_encoded']].value_counts())

In [ ]:
# ── 4.2 One-Hot Encoding — variables sans ordre naturel ───────────
# job, marital, contact, poutcome, month
# → 1 colonne par modalité (0 ou 1)
# drop_first=True pour éviter la redondance (dummy trap)

cols_onehot = ['job', 'marital', 'contact', 'poutcome', 'month']

df_encoded = pd.get_dummies(df_clean, columns=cols_onehot, drop_first=True)

print(f"Colonnes avant encodage : {df_clean.shape[1]}")
print(f"Colonnes après encodage : {df_encoded.shape[1]}")
print()
print("=== Nouvelles colonnes créées ===")
nouvelles_cols = [c for c in df_encoded.columns if c not in df_clean.columns]
print(nouvelles_cols)

In [ ]:
# ── 4.3 Label Encoding — variables avec un ordre naturel ──────────
# education : primary < secondary < tertiary → ordre logique
# default, housing, loan : yes/no → 1/0

df_encoded['education_encoded'] = df_encoded['education'].map({
    'primary': 0,
    'secondary': 1,
    'tertiary': 2
})

df_encoded['default_encoded'] = df_encoded['default'].map({'yes': 1, 'no': 0})
df_encoded['housing_encoded'] = df_encoded['housing'].map({'yes': 1, 'no': 0})
df_encoded['loan_encoded']    = df_encoded['loan'].map({'yes': 1, 'no': 0})

print("=== Vérification education_encoded ===")
print(df_encoded[['education', 'education_encoded']].drop_duplicates().sort_values('education_encoded'))
print()
print("=== Vérification housing_encoded ===")
print(df_encoded[['housing', 'housing_encoded']].drop_duplicates())

### 🔢 4.1, 4.2, 4.3 — Encodage des variables catégorielles

Les algorithmes de Machine Learning ne comprennent que des chiffres.
On convertit donc toutes les variables texte en nombres.

On utilise deux méthodes selon le type de variable :

- **Label Encoding** (variable ordinale — il y a un ordre) :
  - `education` : primary=0, secondary=1, tertiary=2
    (le niveau monte progressivement)
  - `deposit`, `default`, `housing`, `loan` : no=0, yes=1
    (variable binaire, deux choix seulement)

- **One-Hot Encoding** (variable nominale — pas d'ordre) :
  - `job`, `marital`, `contact`, `poutcome`, `month` :
    on crée une colonne par modalité avec 0 ou 1.
    On utilise `drop_first=True` pour éviter la redondance.

> 💼 En langage métier : on traduit chaque information textuelle
> en chiffre pour que le modèle puisse la lire. Pour l'éducation,
> on garde l'ordre (un doctorat vaut plus qu'un bac). Pour le métier,
> pas d'ordre possible — donc une colonne par métier.

In [ ]:
# ── 5.1 Préparation des variables numériques ──────────────────────
# On prépare les colonnes numériques, mais on ne les standardise pas encore.
# La standardisation sera faite après train/test split pour éviter le data leakage.

cols_to_scale = ['age', 'balance', 'duration', 'campaign', 'previous', 'pdays']

df_model = df_encoded.copy()

print("=== Colonnes numériques à standardiser plus tard ===")
print(cols_to_scale)
print()
print("À ce stade, df_model n'est pas encore standardisé.")

In [ ]:
# ── 5.2 Suppression des colonnes texte devenues inutiles ──────────
# On garde uniquement les colonnes numériques pour la modélisation

cols_a_supprimer = ['deposit', 'education', 'default', 'housing', 'loan', 'day']
df_model = df_model.drop(columns=cols_a_supprimer)

print(f"Shape final de df_model : {df_model.shape}")
print()
print("=== Aperçu du DataFrame prêt pour la modélisation ===")
df_model.head(3)

Shape final de df_model : (9556, 39)

=== Aperçu du DataFrame prêt pour la modélisation ===


,age,balance,duration,campaign,pdays,previous,deposit_encoded,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,education_encoded,default_encoded,housing_encoded,loan_encoded
0,1.531002,1.523346,1.941532,-0.812385,-0.489432,-0.372493,1,False,False,False,...,False,False,True,False,False,False,1,0,1,0
1,1.277231,-0.771824,3.173629,-0.812385,-0.489432,-0.372493,1,False,False,False,...,False,False,True,False,False,False,1,0,0,0
2,0.008374,0.451668,2.947503,-0.812385,-0.489432,-0.372493,1,False,False,False,...,False,False,True,False,False,False,1,0,1,0


In [ ]:
# ── 5.3 Vérification finale ────────────────────────────────────────

print("=== Types de colonnes dans df_model ===")
print(df_model.dtypes)
print()

# Vérifier qu'il ne reste aucune colonne texte
cols_texte = df_model.select_dtypes(include='object').columns.tolist()
if len(cols_texte) == 0:
    print("✅ Aucune colonne texte — df_model est 100% numérique")
else:
    print(f"⚠️ Colonnes texte restantes : {cols_texte}")

print()
print(f"✅ df_model prêt : {df_model.shape[0]:,} clients × {df_model.shape[1]} variables")

=== Types de colonnes dans df_model ===
age                  float64
balance              float64
duration             float64
campaign             float64
pdays                float64
previous             float64
deposit_encoded        int64
job_blue-collar         bool
job_entrepreneur        bool
job_housemaid           bool
job_management          bool
job_retired             bool
job_self-employed       bool
job_services            bool
job_student             bool
job_technician          bool
job_unemployed          bool
marital_married         bool
marital_single          bool
contact_telephone       bool
contact_unknown         bool
poutcome_none           bool
poutcome_other          bool
poutcome_success        bool
month_aug               bool
month_dec               bool
month_feb               bool
month_jan               bool
month_jul               bool
month_jun               bool
month_mar               bool
month_may               bool
month_nov               bool
mon

### ⚖️ 5.1, 5.2, 5.3 — Préparation du DataFrame final

**Pourquoi préparer les variables numériques ?**
Certaines variables n’ont pas la même échelle : par exemple, `balance` peut prendre des valeurs beaucoup plus grandes que `campaign`. Si ces variables sont utilisées directement, certains modèles peuvent accorder trop d’importance aux variables avec les plus grands ordres de grandeur.

À ce stade, `df_model` est notre DataFrame final avant modélisation :

* 100% numérique
* aucune colonne texte inutile
* prêt pour la séparation train/test
* prêt pour l’application de la standardisation après le split

Nous ne standardisons pas encore les variables numériques ici afin d’éviter une fuite de données (*data leakage*).

La standardisation sera appliquée ensuite uniquement après la séparation train/test :

* `fit_transform` sur `X_train`
* `transform` sur `X_test`

Ainsi, le jeu de test reste réellement inconnu du modèle.

💼 En langage métier : nous avons mis les dossiers clients dans un format exploitable par un algorithme. Mais nous attendons de séparer les données d’entraînement et de test avant de mettre les variables sur la même échelle, afin d’évaluer le modèle de manière plus fiable.



In [ ]:
# ── 6.1 Séparation Train / Test ────────────────────────────────────
from sklearn.model_selection import train_test_split

# X = les variables explicatives (tout sauf la cible)
# y = la variable cible (deposit_encoded)
X = df_model.drop(columns=['deposit_encoded'])
y = df_model['deposit_encoded']

# 80% pour entraîner le modèle, 20% pour le tester
# random_state=42 pour que tout le monde ait le même résultat
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille totale      : {len(df_model):,} clients")
print(f"Jeu d'entraînement : {len(X_train):,} clients ({round(len(X_train)/len(df_model)*100)}%)")
print(f"Jeu de test        : {len(X_test):,} clients ({round(len(X_test)/len(df_model)*100)}%)")
print()
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

Taille totale      : 9,556 clients
Jeu d'entraînement : 7,644 clients (80%)
Jeu de test        : 1,912 clients (20%)

X_train : (7644, 38)
X_test  : (1912, 38)
y_train : (7644,)
y_test  : (1912,)


In [ ]:
# ── 6.2 Vérification de l'équilibre dans chaque jeu ───────────────
# Important : les deux jeux doivent avoir des proportions similaires
# de souscripteurs (yes/no)

print("=== Répartition deposit dans y_train ===")
print(y_train.value_counts(normalize=True).round(3) * 100)
print()
print("=== Répartition deposit dans y_test ===")
print(y_test.value_counts(normalize=True).round(3) * 100)

=== Répartition deposit dans y_train ===
deposit_encoded
0    52.7
1    47.3
Name: proportion, dtype: float64

=== Répartition deposit dans y_test ===
deposit_encoded
0    51.5
1    48.5
Name: proportion, dtype: float64


### ✂️ 6.1 & 6.2 — Séparation Train / Test

On divise le dataset en deux parties :
- **80% Train** (7 644 clients) : le modèle apprend sur ces données
- **20% Test** (1 912 clients) : on cache ces données au modèle
  pour vérifier qu'il a vraiment appris quelque chose de général

Le `random_state=42` garantit que tout le monde obtient
la même séparation — indispensable pour pouvoir comparer les résultats.

La répartition souscripteurs/non-souscripteurs est similaire
dans les deux jeux : la séparation est équilibrée et représentative.

> 💼 En langage métier : on donne 80% des dossiers au modèle
> pour qu'il s'entraîne, et on garde 20% de côté pour lui faire
> passer un "examen surprise". Si ses prédictions sont bonnes
> sur ces dossiers qu'il n'a jamais vus, on peut lui faire confiance.

In [ ]:
# ── 6.3 Standardisation après Train/Test ──────────────────────────
# Important : on fit le scaler uniquement sur X_train pour éviter le data leakage.

from sklearn.preprocessing import StandardScaler

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("✅ Standardisation appliquée après séparation Train/Test")
print("X_train_scaled :", X_train_scaled.shape)
print("X_test_scaled  :", X_test_scaled.shape)

In [ ]:
# ── 7. Modélisation ───────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

    print("=" * 60)
    print(name)
    print("=" * 60)
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()
    print("Classification report:")
    print(classification_report(y_test, y_pred))

In [ ]:
# ── 8. Comparaison des modèles ────────────────────────────────────

results_df = pd.DataFrame(results)
results_df = results_df.round(3)

results_df

### 📊 Interprétation des modèles

Nous avons comparé deux modèles de classification : la régression logistique et le Random Forest.

La régression logistique est plus simple et plus facile à interpréter.  
Le Random Forest est souvent plus performant car il capture mieux les relations non linéaires entre les variables.

Pour notre problématique métier, le **F1-score** est une métrique importante, car il équilibre la précision et le rappel.  
La banque veut identifier les clients susceptibles de souscrire, mais elle veut aussi éviter de contacter trop de clients peu intéressés.

La variable `duration` est très explicative, mais elle est connue uniquement pendant ou après l’appel. Elle est donc utile pour comprendre la souscription, mais elle est limitée pour prédire à l’avance quels clients contacter avant la campagne.
